In [ ]:
# Acceleration of CNN via Frequency Domain Convolutions

# Phase 1: Environment Setup & Data Pipeline
# In this section, we initialize a deterministic environment by freezing random seeds for strict reproducible benchmarks across different convolution 
# backbones. We also prepare the CIFAR-10 dataset using standard ImageNet normalization statistics to align with pre-trained architectures.

In [1]:
# --- 1. Core Deep Learning Frameworks ---
import torch
import torch.nn as nn
import torch.optim as optim
import torch.fft as fft
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, roc_auc_score, mean_absolute_percentage_error, accuracy_score
import numpy as np
import time
import warnings

warnings.filterwarnings("ignore")

# --- 2. Deterministic Seed Initialization ---
def set_seed(seed=789):
    """Fixes random states across all libraries to ensure objective benchmarks."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(789)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Compute Device: {device}")

# --- 3. CIFAR-10 Dataset & Pipeline Configuration ---
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) # ImageNet statistics for Transfer Learning
])

train_set = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=48, shuffle=True)
test_loader = DataLoader(test_set, batch_size=48, shuffle=False)

Використовується пристрій: cuda


100%|██████████| 170M/170M [00:15<00:00, 11.0MB/s]


In [ ]:
# Phase 2: Frequency Domain Convolution Custom Layer
# Standard spatial convolution scales with a computational complexity of (N^2 dot K^2) where K is the kernel size. 
# According to the Convolution Theorem, element-wise multiplication in the frequency domain equates to linear convolution in the spatial domain, 
# yielding a complexity of (N^2 \log N). We implement a mathematically precise custom layer that caches frozen bank filters and dynamically 
# executes 2D Real-to-Complex FFT operations (`torch.fft.rfft2`).

In [2]:
class AccurateDFTConv2d(nn.Module):
    def __init__(self, original_conv):
        super(AccurateDFTConv2d, self).__init__()
        # Copy frozen weights from the spatial pre-trained layer
        self.weight = nn.Parameter(original_conv.weight.detach(), requires_grad=False)
        if original_conv.bias is not None:
            self.bias = nn.Parameter(original_conv.bias.detach(), requires_grad=False)
        else:
            self.bias = None

        self.padding = original_conv.padding[0]
        self.stride = original_conv.stride[0]

        # Cache allocation for the DFT bank filters spectrum
        self.weight_freq = None

    def forward(self, x):
        # 1. Apply spatial padding to input tensor boundaries
        if self.padding > 0:
            x = torch.nn.functional.pad(x, (self.padding, self.padding, self.padding, self.padding))

        B, I, H, W = x.shape
        O, _, KH, KW = self.weight.shape

        # 2. Compute 2D DFT for the padded frozen filter banks (Single execution cache)
        if self.weight_freq is None or self.weight_freq.shape[-2] != H:
            weight_padded = torch.zeros((O, I, H, W), device=x.device, dtype=x.dtype)
            weight_padded[:, :, :KH, :KW] = self.weight
            self.weight_freq = torch.fft.rfft2(weight_padded)

        # 3. Transform spatial feature maps into the frequency domain via Real FFT
        X_freq = torch.fft.rfft2(x)

        # 4. Perform convolution via tensor contraction (einsum product multiplication)
        Y_freq = torch.einsum('bihw,oihw->bohw', X_freq, self.weight_freq)

        # 5. Transform back to spatial domain via Inverse Real FFT
        out = torch.fft.irfft2(Y_freq, s=(H, W))

        # 6. Crop circular convolution boundary artifacts to retrieve exact linear convolution equivalent
        out = out[:, :, KH-1:, KW-1:]

        # 7. Inject bias vector and apply spatial striding operations
        if self.bias is not None:
            out += self.bias.view(1, -1, 1, 1)
        if self.stride > 1:
            out = out[:, :, ::self.stride, ::self.stride]

        return out

In [ ]:
# Phase 3: Backbone Adaptation & Transfer Learning Setup
# We load a pre-trained VGG16 model architecture. Without altering the frozen model weights, we modify the backbone structure to swap out 
# standard spatial convolutions with our custom `AccurateDFTConv2d` layers while padding smaller kernels to verify complexity scaling. 
# We also adapt the linear classification head for CIFAR-10.

In [3]:
def build_vgg_models(kernel_size=3):
    """
    Constructs VGG16 models comparing classic spatial vs. frequency domain convolutions.
    Artificially pads kernels to evaluate complexity performance over varying filter bounds.
    """
    def modify_model(model, use_dft=False):
        new_features = []
        for layer in model.features:
            if isinstance(layer, nn.Conv2d):
                # Instantiate a new convolutional target with specified sizing boundaries
                new_conv = nn.Conv2d(
                    layer.in_channels, layer.out_channels,
                    kernel_size=kernel_size, padding=kernel_size // 2
                )
                with torch.no_grad():
                    new_conv.weight.fill_(0)
                    center = kernel_size // 2
                    # Copy the pre-trained 3x3 kernels directly into the center of the padded kernel
                    if kernel_size >= 3:
                        new_conv.weight[:, :, center-1:center+2, center-1:center+2] = layer.weight
                    if layer.bias is not None:
                        new_conv.bias.copy_(layer.bias)

                # Swap out layers based on the target execution strategy
                if use_dft:
                    new_features.append(AccurateDFTConv2d(new_conv))
                else:
                    new_features.append(new_conv)
            else:
                new_features.append(layer)

        model.features = nn.Sequential(*new_features)

        # Freeze the backbone feature extractor completely (Static filter bank)
        for param in model.features.parameters():
            param.requires_grad = False

        # Adapt final classification dense layer to map 10 CIFAR-10 output nodes
        model.classifier[6] = nn.Linear(4096, 10)
        return model.to(device)

    # Instantiate twin network setups utilizing VGG16 weights
    orig_model = modify_model(models.vgg16(weights='IMAGENET1K_V1'), use_dft=False)
    dft_model = modify_model(models.vgg16(weights='IMAGENET1K_V1'), use_dft=True)

    return orig_model, dft_model

In [ ]:
# Phase 4: Training & Evaluation Framework
# We establish an evaluation tracking routine calculating F1-score (macro-averaged), ROC-AUC (One-vs-Rest), and 
# standard classification Accuracy across the datasets. The training routine logs duration explicitly per epoch execution.

In [4]:
def evaluate_model(model, loader):
    """Computes comprehensive evaluation metrics over the target DataLoader."""
    model.eval()
    all_preds, all_targets, all_probs = [], [], []

    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    f1 = f1_score(all_targets, all_preds, average='macro')
    auc = roc_auc_score(all_targets, all_probs, multi_class='ovr')
    accuracy = accuracy_score(all_targets, all_preds)

    return f1, auc, accuracy

def train_epoch(model, name):
    """Executes a single epoch run over the classification head, tracking wall time."""
    optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    start_time = time.time()
    model.train()

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

    duration = time.time() - start_time
    f1, auc, acc = evaluate_model(model, test_loader)

    print(f"| {name:<20} | {duration:<10.2f} | {f1:<8.4f} | {auc:<8.4f} | {acc:<8.4f} |")
    return duration, f1, auc, acc

In [ ]:
# Phase 5: Executing Comparative Experiments & Isolated Layer Micro-Benchmarks
# We run an end-to-end epoch experiment across small (3 times) and larger (7 times) kernel boundaries. 
# Finally, we execute an isolated layer micro-benchmark over a series of expansive kernel topologies 
# on standard feature maps to capture the exact empirical crossover point where frequency transformations outperform spatial convolution matrices.

In [7]:
kernel_sizes = [3, 7]
summary_results = []

print("=" * 75)
print(f"| {'Architecture (Kernel)':<20} | {'Time (s)':<10} | {'F1-score':<8} | {'ROC-AUC':<8} | {'Accuracy':<8} |")
print("=" * 75)

for k in kernel_sizes:
    orig_vgg, dft_vgg = build_vgg_models(kernel_size=k)

    set_seed(789)
    orig_time, f1_o, auc_o, acc_o = train_epoch(orig_vgg, f"Original (K={k}x{k})")

    set_seed(789)
    dft_time, f1_d, auc_d, acc_d = train_epoch(dft_vgg, f"DFT-based (K={k}x{k})")
    print("-" * 75)

    summary_results.append({
        'Size': k, 'Orig_Time': orig_time, 'DFT_Time': dft_time,
        'Orig_Acc': acc_o, 'DFT_Acc': acc_d
    })

# --- 1. DISPLAY SYSTEM SUMMARY TABLE ---
print("\n" + " " * 20 + "SUMMARY PERFORMANCE OVERVIEW")
print("-" * 65)
print(f"{'Kernel Size':<10} | {'Orig Time (s)':<13} | {'DFT Time (s)':<12} | {'Orig Acc':<10} | {'DFT Acc':<10}")
print("-" * 65)
for r in summary_results:
    print(f"{r['Size']}x{r['Size']:<8} | {r['Orig_Time']:<13.2f} | {r['DFT_Time']:<12.2f} | {r['Orig_Acc']:<10.4f} | {r['DFT_Acc']:<10.4f}")

# --- 2. LAYER ISOLATED MICRO-BENCHMARK ---
print("\n" + " " * 10 + "MICRO-BENCHMARK: SPATIAL CONVOLUTION VS. DFT CONVOLUTION")
print("-" * 60)
print(f"{'Kernel Sizing':<15} | {'Spatial Conv (s)':<20} | {'DFT Conv (s)':<15}")
print("-" * 60)

test_kernels = [5, 15, 25, 35]
dummy_input = torch.randn(16, 64, 128, 128).to(device) # High-dimension feature map simulation

for tk in test_kernels:
    conv_spatial = nn.Conv2d(64, 64, kernel_size=tk, padding=tk//2).to(device)
    conv_dft = AccurateDFTConv2d(conv_spatial).to(device)

    # Warmup and test spatial matrix multiplication pipeline
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(10): _ = conv_spatial(dummy_input)
    torch.cuda.synchronize()
    t_spatial = (time.time() - start) / 10

    # Warmup and test Real FFT frequency multiplication pipeline
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(10): _ = conv_dft(dummy_input)
    torch.cuda.synchronize()
    t_dft = (time.time() - start) / 10

    print(f"{f'{tk}x{tk}':<15} | {t_spatial:<20.5f} | {t_dft:<15.5f}")

| Модель (Kernel)      | Час (сек)  | F1-score | ROC-AUC  | Accuracy |
| Original (K=3x3)     | 54.52      | 0.6118   | 0.9112   | 0.6129   |
| DFT-based (K=3x3)    | 79.26      | 0.5201   | 0.8967   | 0.5283   |
---------------------------------------------------------------------------
| Original (K=7x7)     | 122.65     | 0.5942   | 0.9217   | 0.5977   |
| DFT-based (K=7x7)    | 127.99     | 0.5045   | 0.8872   | 0.5075   |
---------------------------------------------------------------------------

                    ПІДСУМКОВА ТАБЛИЦЯ
-----------------------------------------------------------------
Kernel     | Orig Time    | DFT Time     | Orig Acc   | DFT Acc   
-----------------------------------------------------------------
3x3        | 54.52        | 79.26        | 0.6129     | 0.5283    
7x7        | 122.65       | 127.99       | 0.5977     | 0.5075    

          МІКРО-БЕНЧМАРК: ПРОСТОРОВА ЗГОРТКА ПРОТИ ДПФ
------------------------------------------------------------
Ker